# 02 · openFDA Enforcement API (food · drug · device)

**Agency:** U.S. Food & Drug Administration
**Endpoints:** `https://api.fda.gov/{food|drug|device}/enforcement.json` — same schema for all three
**Auth:** none needed for light use; a free API key raises the daily limit  ·  **Updated:** weekly

> ⚠️ FDA's own disclaimer says this data **should not be used to issue public alerts** or to track a recall's lifecycle, and `status` is **not updated** after classification. Design consequence: food/drug results are a *lookup* with a "confirm on FDA.gov" link, not a push alert.

> Saved samples: 10 food, 5 drug, 5 device — the newest reports (Oct 14 2026).

In [1]:
import sys, json, re
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # so we can import recall_explore from the repo root
import pandas as pd
from recall_explore import fetch, load_samples, find_ids, coverage, normalize_model
pd.set_option("display.max_colwidth", 80)

LIVE = False   # True = call the real API (needs internet). False = use the 20 saved samples.

## 1 · Fetch
openFDA uses `search=`, `sort=`, `limit=` (max 1000) and `skip=` for paging. Example query language: `search=report_date:[20260901+TO+20261020]`.
`meta.results.total` tells you how many records exist.

In [2]:
if LIVE:
    recalls = []
    for ep, n in [("food", 10), ("drug", 5), ("device", 5)]:
        res = fetch("openfda", endpoint=ep, sort="report_date:desc", limit=n)
        print(ep, "total records:", res["meta"]["results"]["total"])
        recalls += [{**r, "_endpoint": ep} for r in res["results"]]
else:
    recalls = load_samples("openfda")
print(len(recalls), "records loaded")
# Totals seen on 2026-10-20: food 29,505 · drug 18,043 · device 40,300

20 records loaded


## 2 · Anatomy of one record

In [3]:
r = next(x for x in recalls if x["recall_number"] == "H-0016-2027")
for k, v in r.items():
    print(f"{k:<28}", json.dumps(v, ensure_ascii=False)[:110])

status                       "Ongoing"
city                         "Clermont"
state                        "FL"
country                      "United States"
classification               "Class I"
openfda                      {}
product_type                 "Food"
event_id                     "99872"
recalling_firm               "A&P CREATIONS"
address_1                    "2725 Armstrong Ave"
address_2                    "N/A"
postal_code                  "34714-8117"
voluntary_mandated           "Voluntary: Firm initiated"
initial_firm_notification    "Press Release"
distribution_pattern         "Distributed nationwide (US)."
recall_number                "H-0016-2027"
product_description          "X10 Natural Enhancement Supplement, 0.5 fl oz/sachet (15 mL), 7 sachets/box, UPC: 7708075247094, Manufactured
product_quantity             "1,283 retail units"
reason_for_recall            "Product contains undeclared Sildenafil and Tadalafil."
recall_initiation_date       "20260914"
center

## 3 · Field coverage
Unlike CPSC, almost every field is filled — but the identifiers are packed into **two free-text fields**:
- `product_description` → brand, size, **UPC**, sometimes model / catalog number, NDC
- `code_info` → **lot numbers**, expiry / best-by dates, serials, UDI

In [4]:
coverage(recalls, ["recall_number", "recalling_firm", "product_description", "code_info",
                   "reason_for_recall", "classification", "status", "distribution_pattern",
                   "product_quantity", "report_date", "openfda"])

,filled,of,pct
field,,,
recall_number,19,20,95
recalling_firm,20,20,100
product_description,20,20,100
code_info,19,20,95
reason_for_recall,20,20,100
classification,20,20,100
status,20,20,100
distribution_pattern,20,20,100
product_quantity,19,20,95


`openfda` is `{}` for nearly every record. It's only populated for some **drugs** (it adds brand names, NDCs and even a UPC list). Don't rely on it.

## 4 · Regex first pass on the two text fields

In [5]:
rows = []
for x in recalls:
    pid = find_ids(x["product_description"])
    cid = find_ids(x["code_info"])
    rows.append({"recall": x["recall_number"], "type": x["_endpoint"],
                 "upc (desc)": len(pid["upcs"]), "ndc": len(pid["ndcs"]),
                 "model": ", ".join(pid["models"])[:30] or "—",
                 "lots (code_info)": len(cid["lots"]), "serials": len(cid["serials"]),
                 "udi": len(cid["udis"]), "date codes": len(cid["date_codes"]),
                 "code_info": x["code_info"][:60]})
t = pd.DataFrame(rows).set_index("recall")
t

,type,upc (desc),ndc,model,lots (code_info),serials,udi,date codes,code_info
recall,,,,,,,,,
Z-0074-2027,device,0,0,S3065,1,0,1,3,UDI: 10724995220300 Lot numbers: 260810: expiration date - 0
Z-0078-2027,device,0,0,—,0,4,3,0,"1. Model Number: 728333 (U.S.); UDI-DI: 1. 00884838101111, 2"
Z-0085-2027,device,0,0,—,1,1,1,0,"UDI-DI: 00885825003838. Lot: 5640704, Serial: 5097742, Distr"
Z-0087-2027,device,0,0,—,1,0,0,0,Lot Numbers: 05425080001 05426040001
Z-0088-2027,device,0,0,MMT-8062,0,3,2,0,"MMT-8072 Lot Code: Serial numbers: FB00015839, FB00016511, F"
D-0006-2027,drug,6,3,—,0,0,0,5,"Lot: a) LT25B01, Exp.: 02/28/2027; b) LT25D02, Exp.: 04/30/2"
D-0014-2027,drug,0,1,—,1,0,0,1,"Lot#: 5KA0405, Exp 12/31/2026."
D-0016-2027,drug,0,1,—,1,0,0,1,Lot JNYJ1208; Exp. 10/30/2027
D-0021-2027,drug,0,0,—,1,0,0,1,"LOT: 08032026@24, BUD: 09/17/2026."


In [6]:
for ep in ["food", "drug", "device"]:
    s = t[t.type == ep]
    print(f"{ep:<7} UPC in desc: {(s['upc (desc)']>0).sum()}/{len(s)}   "
          f"lot found: {(s['lots (code_info)']>0).sum()}/{len(s)}   "
          f"date code: {(s['date codes']>0).sum()}/{len(s)}")

food    UPC in desc: 8/10   lot found: 5/10   date code: 4/10
drug    UPC in desc: 1/5   lot found: 4/5   date code: 5/5
device  UPC in desc: 0/5   lot found: 3/5   date code: 1/5


## 5 · The regex is not enough — look at the lists
The lot regex grabs the **first** lot after the word "Lot", but many records list a dozen. Compare what's really there:

In [7]:
for n in ["D-0006-2027", "Z-0074-2027"]:
    x = next(y for y in recalls if y["recall_number"] == n)
    print(n, "| regex lots:", find_ids(x["code_info"])["lots"])
    print("   real code_info:", x["code_info"][:400], "\n")

D-0006-2027 | regex lots: []
   real code_info: Lot: a) LT25B01, Exp.: 02/28/2027; b) LT25D02, Exp.: 04/30/2027; LT25K01, Exp.: 09/30/2027; c) LT25D02, Exp.: 04/30/2027; LT25K01, Exp.: 09/30/2027; d) LT25D02, Exp.: 04/30/2027; LT25M02, Exp.: 10/31/2027; e) LT25B01, Exp.: 02/28/2027; LT25D01, Exp.: 04/30/2027; LT25D02, Exp.: 04/30/2027; LT25K01, Exp.: 09/30/2027; LT25M01, Exp.: 10/31/2027; LT25M02, Exp.: 10/31/2027; LT25N01, Exp.: 11/30/2027; LT 

Z-0074-2027 | regex lots: ['260810']
   real code_info: UDI: 10724995220300 Lot numbers: 260810: expiration date - 08/10/2026 260810A: expiration date - 08/10/2026 261106: expiration date - 11/06/2026 270116: expiration date - 01/16/2027 



`D-0006-2027` (dry-eye drops) is the perfect hard case: **9 store brands** (a–i) in `product_description`, each with its own UPC/NDC, and `code_info` maps each letter to its own lots and expiry dates.
An LLM with a strict JSON schema has to rebuild that mapping: `[{brand, upc, lots:[{lot, exp}]}]`.

## 6 · Formats you'll need to normalise

In [8]:
examples = {
  "UPC printed with spaces":   "UPC: 0 75465 11111 0",
  "UPC with ###### wildcard":  "UPC 2 67240 ######  (store-packed weighed item)",
  "Lot with @ suffix":          "LOT: 08032026@24, BUD: 09/17/2026",
  "Date range, not lots":       "Best If Used By Dates from August 14th, 2026 to September 25th, 2026.",
  "'All dates through'":        "Best By: all dates through 3/30/2027",
  "No codes at all":            "no codes",
  "Device serial list":         "Serial numbers: FB00015839, FB00016511, …",
  "Device UDI-DI (GTIN-14)":    "UDI-DI: 00885825003838",
}
for k, v in examples.items():
    print(f"{k:<28} {v:<60} -> upc {find_ids(v)['upcs']}")

UPC printed with spaces      UPC: 0 75465 11111 0                                         -> upc ['075465111110']
UPC with ###### wildcard     UPC 2 67240 ######  (store-packed weighed item)              -> upc []
Lot with @ suffix            LOT: 08032026@24, BUD: 09/17/2026                            -> upc []
Date range, not lots         Best If Used By Dates from August 14th, 2026 to September 25th, 2026. -> upc []
'All dates through'          Best By: all dates through 3/30/2027                         -> upc []
No codes at all              no codes                                                     -> upc []
Device serial list           Serial numbers: FB00015839, FB00016511, …                    -> upc []
Device UDI-DI (GTIN-14)      UDI-DI: 00885825003838                                       -> upc []


## 7 · Other things worth knowing
- Dates are compact strings `YYYYMMDD` (`"20261014"`). Parse them.
- One **event** (`event_id`) → many `recall_number`s (one per product). E.g. Carolina Infusion event 99867 has 4+ drug recall numbers. Group by `event_id` for the UI.
- `recall_number` can literally be `"N/A"` while a recall is *Not Yet Classified* → use `event_id` + product as a fallback key.
- `classification`: **Class I** = serious health risk / death, II = temporary harm, III = unlikely harm. Great severity signal for alert priority.
- `distribution_pattern` is free text ("FL and NH", "Nationwide") — could power "only alert users in affected states".
- Device recalls are mostly hospital equipment (CT scanners, catheters). Only some (insulin pumps, walkers, home test kits) matter to consumers → filter.

## ✅ Takeaways for Recall Radar
1. **Food is the UPC-rich source** (UPC in 8/10 food descriptions) → barcode scan → exact match is realistic here. Watch for store-packed codes starting with `2` (variable weight, e.g. `2 67240 ######`) — those won't match a shelf barcode.
2. Lots and dates are in `code_info` as messy lists → LLM extraction to `lot_ranges` JSON, keep `code_info` raw.
3. Use `classification` for severity and `event_id` for grouping.
4. Respect FDA's disclaimer: lookup + "verify on FDA.gov", not authoritative alerts. Log this in Decisions.